# EDA2 — 2023년이 왜 특이한가

지난 실험(`modeling/baseline_catboost.py`의 rolling OOT)에서 2019~2022로 학습한 모델이 2023 예측에서
베이스라인(그냥 평균 찍기)보다도 못했다 (Brier 0.2531 > r(1-r) 0.25). "2023이 특이한 연도"라는 가설을
직접 확인한다.

함수는 [eda2.py](eda2.py). 원본 컬럼 정의는 [data_description.md](../../data/data_description.md) 참고.

## 확인할 것
1. 타겟(성공률) 자체의 연도별 추세에 특이점이 있는가
2. `game_type=F`의 의미가 실제로 뒤집히는가
3. 입력 피처 분포 자체가 2023에 달라지는가 (covariate shift)
4. 처음 보는 투수/타자 비율이 2023에 튀는가
5. 팀 구성이 바뀌는가

In [1]:
from eda2 import *  # noqa: F403
import pandas as pd

df = load("train.csv")
print(f"shape={df.shape}")

shape=(1475092, 49)


## 1. 연도별 성공률 추세

In [2]:
yearly_target_summary(df)

,success_rate,count,yoy_delta
season,,,
2019,0.5647,237413,NaN
2020,0.5327,244087,-0.0320
2021,0.5328,247088,0.0000
2022,0.5289,247472,-0.0038
2023,0.5000,245525,-0.0290
2024,0.4861,253507,-0.0139


전체 평균은 2019(0.565)→2024(0.486)로 매년 완만하게 떨어진다. 2023의 전년 대비 하락폭(-0.029)이 2022(-0.004)보다는 크지만, 2019→2020(-0.032)도 비슷한 크기라 **이 표만 보면 2023이 유별나 보이지 않는다.** 진짜 차이는 다음 절에서 나온다.

## 2. `game_type` x 연도 성공률

In [3]:
regime_crosstab(df, "game_type")

mean          count        
game_type       F       R      F       R
season                                  
2019       0.6892  0.5495  25786  211627
2020       0.5878  0.5269  23213  220874
2021       0.7038  0.5128  25861  221227
2022       0.7087  0.5037  30448  217024
2023       0.4729  0.5031  25686  219839
2024       0.4593  0.4897  30010  223497

**여기서 뒤집힌다.** `game_type=F`의 성공률:

```
2019: 0.689   2020: 0.588   2021: 0.704   2022: 0.709   2023: 0.473   2024: 0.459
```

2019-2022엔 0.59-0.71로 계속 높다가 **2023에 0.47로 떨어져서** 2024까지 낮게 유지된다. 반면 `R`은 0.55→0.49로 서서히 drift할 뿐 급변이 없다.

`F`는 전체의 11%뿐이라 이 반전이 1절의 전체 평균(성공률 추세)에는 크게 안 드러난다. 하지만 `game_type`을 강하게 학습한 모델(우리 CatBoost가 이걸 변수중요도 1위로 씀)한테는 치명적이다 — 2019~2022로 학습하면 "F=고성공"을 배우는데, 2023부터는 정반대라서 자신 있게 틀리게 된다. 이게 바로 rolling OOT에서 2023 폴드가 베이스라인보다 못했던 이유다.

이건 우리 EDA에서 직접 확인한 반전 패턴이다.

## 3. 입력 피처 분포 자체가 바뀌었나 (covariate shift)

In [4]:
feature_shift_2023_vs_rest(df).head(10)

,rest_mean,y2023_mean,diff,diff_pct
score_diff_home,-0.2387,-0.0111,0.2276,0.9536
asof_pitcher_n,2417.8180,3880.7744,1462.9564,0.6051
asof_batter_n,3001.4517,4619.9568,1618.5051,0.5392
run_top_before,2.6029,2.3324,-0.2705,-0.1039
asof_pitcher_reverse_rate,0.2117,0.2330,0.0214,0.1010
score_diff_pitcher_team,0.0593,0.0641,0.0048,0.0813
run_total_before,4.9670,4.6537,-0.3133,-0.0631
asof_pitcher_middle_rate,0.1407,0.1475,0.0068,0.0481
asof_batter_middle_rate,0.1395,0.1458,0.0063,0.0455
li,0.9747,1.0178,0.0431,0.0442


상위권에 있는 것들을 뜯어보면 대부분 진짜 이상 신호가 아니다.

- `score_diff_home`: diff_pct 95%로 제일 커 보이지만, `rest_mean`(-0.239)과 `y2023_mean`(-0.011) 둘 다 0에 가까운 값이라 **분모가 작아서 퍼센트가 부풀려진 것뿐**이다 (절대 차이는 0.23점 정도로 작음).
- `asof_pitcher_n`/`asof_batter_n`: 2023이 60%/54% 높게 나오는데, 이건 **누적 카운트 피처라 시간이 지날수록 자연히 커지는 게 정상**이다 (2024는 더 높을 것). 2023만의 특이점이 아니라 전체 기간에 걸친 자연스러운 증가 추세.

즉 입력 피처 분포 자체는 2023에 별다른 이상이 없다 — **넓은 분포 변화(covariate shift)가 아니라, `game_type` 한 변수의 타겟과의 관계만 좁고 정확하게 뒤집힌 concept drift**라는 뜻이다.

## 4. 처음 보는 투수/타자 비율

In [5]:
print("[투수]")
print(new_entity_ratio_by_year(df, "pitcher_id"))
print("\n[타자]")
print(new_entity_ratio_by_year(df, "batter_id"))

[투수]


        unique_ids  new_ids  new_ratio
season                                
2019           355      355     1.0000
2020           356      110     0.3090
2021           386       95     0.2461
2022           390       88     0.2256
2023           382       63     0.1649
2024           391       81     0.2072

[타자]


        unique_ids  new_ids  new_ratio
season                                
2019           400      400     1.0000
2020           371       95     0.2561
2021           398       88     0.2211
2022           403       88     0.2184
2023           397       71     0.1788
2024           424       88     0.2075


2023의 신규 투수 비율(0.165)·신규 타자 비율(0.179)은 오히려 다른 해(0.22~0.31)보다 **더 낮다**. 선수 물갈이가 심해서 생긴 문제가 아니라는 뜻.

## 5. 팀 구성 변화

In [6]:
team_composition_by_year(df, "pitcher_team_id").loc[[2021, 2022, 2023, 2024]].T

season,2021,2022,2023,2024
pitcher_team_id,,,,
12,0.0976,0.1017,0.0995,0.1002
13,0.1386,0.1454,0.1409,0.1469
14,0.0953,0.0973,0.0968,0.0950
15,0.0960,0.0920,0.0941,0.0919
16,0.0934,0.0917,0.0918,0.0925
17,0.0999,0.1010,0.0986,0.0983
18,0.0933,0.0933,0.0930,0.0900
19,0.0936,0.0923,0.0925,0.0911
20,0.0888,0.0857,0.0865,0.0897


팀별 비중이 2021~2024 내내 소수점 둘째 자리 수준에서만 미세하게 바뀐다. 로스터/리그 구성이 바뀐 것도 아니다.

## 결론

2023은 데이터가 전반적으로 이상해진 해가 아니다. **`game_type=F`라는 딱 하나의 변수가 타겟과 맺는 관계가 정확히 그 시점에 반전됐고**, 그 외 모든 것(피처 분포, 선수 구성, 팀 구성)은 평소와 다를 게 없다. 그래서 이건 모델링으로 사전에 감지할 수 없는 종류의 변화다 — 2019~2022 데이터만 봐서는 이 반전이 온다는 어떤 단서도 없다 (out-of-distribution).

실전에서 의미하는 바: 우리가 실제로 제출하는 모델은 2019~2024 전체로 학습하기 때문에 이미 이 반전을 한 번 겪은 뒤의 데이터를 갖고 있다. 문제는 **2025에 또 다른 반전이 있을 수 있는가**인데, 이건 알 수 없다. 대응 전략은 새로운 반전을 예측하는 게 아니라, 모델이 특정 변수(특히 `game_type`처럼 한 번 뒤집힌 전력이 있는 변수)에 과도하게 의존하지 않도록 **최근 연도에 더 가중치를 주는 것**이다 — 다음 노트북([recency_weighted_catboost.ipynb](../modeling/recency_weighted_catboost.ipynb))에서 시도한다.